# Notebook 1 (ASDiv) — SG Data Generation with Qwen2.5-3B-Instruct

**Dataset changed from GSM8K → ASDiv**
- ASDiv has ~2,300 diverse grade-school math problems (add/subtract/multiply/divide/mixed)
- HuggingFace ID: `EleutherAI/asdiv`
- Only change from GSM8K version: Cell 5 (dataset loading) and Cell 10 (field names in loop)

**Expected valid rate: 70-85%**

**Memory: ~6GB float16 on P100 — safe without 4-bit**

In [ ]:
# ── CELL 1: Install ───────────────────────────────────────────
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")

In [ ]:
# ── CELL 2: HuggingFace Login ─────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets  = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
print("HuggingFace login successful.")

In [ ]:
# ── CELL 3: Imports + GPU Check ───────────────────────────────
import os, json, re, random, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/sg_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── CELL 4: Config ────────────────────────────────────────────
# num_samples: 50 = quick test | 2000 = full run
# ASDiv has ~2,300 problems total — 2000 is a safe full-run number

CONFIG = {
    "model_name"         : "Qwen/Qwen2.5-3B-Instruct",
    "num_samples"        : 2000,      # ← change to 50 for a quick test first
    "random_seed"        : 42,
    "max_new_tokens"     : 150,
    "temperature"        : 0.1,
    "do_sample"          : True,
    "batch_size"         : 8,
    "max_words_per_step" : 20,
    "raw_output_file"    : f"{OUTPUT_DIR}/sg_raw.jsonl",
    "valid_output_file"  : f"{OUTPUT_DIR}/sg_valid.jsonl",
    "checkpoint_file"    : f"{OUTPUT_DIR}/checkpoint.json",
    "report_file"        : f"{OUTPUT_DIR}/generation_report.json",
    "save_every"         : 50,
}

print("Config:")
for k, v in CONFIG.items():
    print(f"  {k:22s}: {v}")

In [ ]:
# ── CELL 5: Load ASDiv ────────────────────────────────────────
# ASDiv fields: body, question, answer, formula, type
# We combine body + question into one string (same shape as GSM8K's 'question').
# Answer is a string like "12" or "3.5" or "12 (dozen)" — we strip the unit suffix.

print("Loading ASDiv from HuggingFace...")
asdiv = load_dataset("EleutherAI/asdiv", trust_remote_code=True)

# ASDiv only ships a single 'validation' split (~2,305 problems)
train_data = list(asdiv["validation"])

print(f"ASDiv total : {len(train_data)} problems")
print(f"Features    : {list(asdiv['validation'].features.keys())}")
print(f"\nRaw example:")
for k, v in train_data[0].items():
    print(f"  {k}: {v}")


def asdiv_question(item):
    """Merge body + question into one string — equivalent to GSM8K 'question'."""
    return item["body"].strip().rstrip(".") + " " + item["question"].strip()


def asdiv_answer(item):
    """
    Extract a clean numeric string from ASDiv's answer field.
    ASDiv answers look like: '12', '3.5', '12 (dozen)', '2 hours'
    We keep only the leading number and normalise 5.0 -> '5'.
    """
    raw = str(item["answer"]).strip()
    m   = re.match(r"(-?[\d\.]+)", raw)
    ans = m.group(1) if m else raw
    try:
        f = float(ans)
        return str(int(f)) if f == int(f) else str(round(f, 4))
    except Exception:
        return ans


random.seed(CONFIG["random_seed"])
num          = min(CONFIG["num_samples"], len(train_data))
indices      = random.sample(range(len(train_data)), num)
sampled_data = [train_data[i] for i in indices]

print(f"\nSampled    : {len(sampled_data)}")
print(f"\nExample question : {asdiv_question(sampled_data[0])}")
print(f"Example answer   : {asdiv_answer(sampled_data[0])}")

In [ ]:
# ── CELL 6: Load Qwen2.5-3B-Instruct ────────────────────────
print(f"Loading {CONFIG['model_name']}...")
print("Expected GPU memory: ~6GB in float16")

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code = True,
)
tokenizer.padding_side = "left"    # critical for batch generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    dtype      = torch.float16,
    device_map = "auto",
)
model.eval()

used_gb  = torch.cuda.memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nModel loaded!")
print(f"GPU used  : {used_gb:.2f} GB / {total_gb:.1f} GB")
print(f"Headroom  : {total_gb - used_gb:.1f} GB remaining")
print(f"Padding   : {tokenizer.padding_side} ✅")

if used_gb > 10:
    print("⚠️ WARNING: High memory. Reduce batch_size to 4 in Config if OOM.")
else:
    print("✅ Memory looks good.")

In [ ]:
# ── CELL 7: Prompt ────────────────────────────────────────────
# Identical to GSM8K version — prompt format does not depend on dataset.

SYSTEM_PROMPT = """You are a math problem decomposition assistant.
Your job: break a math problem into 2-5 numbered solution steps.

RULES:
1. Use format: Step 1: ... Step 2: ... etc.
2. Each step must be under 15 words.
3. Do NOT perform calculations or write numbers from computation.
4. Do NOT write the final answer.
5. Write ONLY the steps. Stop immediately after the last step.

EXAMPLE:
Problem: John earns $10/hour and works 8 hours. What does he earn?
Step 1: Identify the hourly rate and total hours worked.
Step 2: Multiply the hourly rate by the number of hours.
Step 3: The result is the total earnings."""

USER_TEMPLATE = """Problem: {question}"""


def build_prompt(question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_TEMPLATE.format(question=question)},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize              = False,
        add_generation_prompt = True,
    )


print("Prompt preview:")
print(build_prompt(asdiv_question(sampled_data[0])))

In [ ]:
# ── CELL 8: Cleaning + Filtering ─────────────────────────────
# Identical to GSM8K version.

def clean_sg_output(text: str) -> str:
    """Extract only Step lines. Stop at first non-step line."""
    lines        = text.strip().split("\n")
    clean_lines  = []
    step_started = False

    for line in lines:
        line = line.strip()
        if not line:
            continue
        is_step = bool(re.match(
            r"^(Step\s*\d+[:.)]|\d+[.)]\s)", line, re.IGNORECASE
        ))
        if is_step:
            step_started = True
            trimmed = re.split(r"\.\s+[A-Z]", line)[0]
            if not trimmed.endswith("."):
                trimmed += "."
            clean_lines.append(trimmed)
        elif step_started:
            break

    return "\n".join(clean_lines)


def extract_steps(text: str) -> list:
    return [
        l.strip() for l in text.strip().split("\n")
        if re.match(r"^(Step\s*\d+[:.)]|\d+[.)]\s)", l.strip(), re.IGNORECASE)
    ]


def step_body(step: str) -> str:
    return re.sub(
        r"^(Step\s*\d+[:.)]|\d+[.)]\s)", "", step, flags=re.IGNORECASE
    ).strip()


def contains_calculation(text: str) -> bool:
    patterns = [
        r"\d+\s*[\+\-\×\÷\*\/]\s*\d+",
        r"=\s*\$?\d+",
        r"the answer is\s+\d+",
        r"\$\s*\d+\.?\d*",
        r"\\frac", r"\\times", r"\\left",
        r"\\\(", r"\\\)",
    ]
    return any(re.search(p, text, re.IGNORECASE) for p in patterns)


def steps_sequential(steps: list) -> bool:
    for i, step in enumerate(steps):
        m = re.match(r"^(?:Step\s*)(\d+)", step, re.IGNORECASE)
        if m and int(m.group(1)) != i + 1:
            return False
    return True


def steps_concise(steps: list, max_words: int) -> bool:
    return all(len(step_body(s).split()) <= max_words for s in steps)


def is_valid_sg(text: str) -> tuple:
    if not text or len(text.strip()) < 15:
        return False, "too_short"

    steps = extract_steps(text)

    if len(steps) < 2:                                        return False, "too_few_steps"
    if len(steps) > 5:                                        return False, "too_many_steps"
    if contains_calculation(text):                            return False, "contains_calculation"
    if not steps_sequential(steps):                          return False, "non_sequential_steps"
    if not steps_concise(steps, CONFIG["max_words_per_step"]): return False, "step_too_long"

    action_verbs = [
        "calculate", "determine", "identify", "find", "compute",
        "add", "subtract", "multiply", "divide", "sum", "count",
        "check", "compare", "convert", "evaluate", "use", "total",
        "measure", "estimate", "figure", "apply"
    ]
    if not any(v in text.lower() for v in action_verbs):
        return False, "no_action_verb"

    return True, "ok"


print("Filters loaded. Running sanity check...")
good = """Step 1: Identify the number of items in each group.
Step 2: Multiply the number of groups by items per group.
Step 3: Add any remaining items to get the total."""
v, r = is_valid_sg(good)
steps = extract_steps(good)
print(f"Test SG → valid={v}, reason={r}, steps={len(steps)}")
for s in steps:
    print(f"  [{len(step_body(s).split()):2d}w] {s}")

In [ ]:
# ── CELL 9: Generation + Single Test ─────────────────────────

def generate_sg_batch(questions: list) -> list:
    prompts = [build_prompt(q) for q in questions]

    inputs = tokenizer(
        prompts,
        return_tensors = "pt",
        padding        = True,
        truncation     = True,
        max_length     = 512,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = CONFIG["max_new_tokens"],
            temperature        = CONFIG["temperature"],
            do_sample          = CONFIG["do_sample"],
            pad_token_id       = tokenizer.eos_token_id,
            eos_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.2,
        )

    input_len = inputs["input_ids"].shape[1]
    results   = []
    for output in outputs:
        raw   = tokenizer.decode(output[input_len:], skip_special_tokens=True).strip()
        clean = clean_sg_output(raw)
        results.append((raw, clean))
    return results


# ── Single test ──
print("=" * 55)
print("SINGLE SAMPLE TEST")
print("=" * 55)

test_q   = asdiv_question(sampled_data[0])
test_out = generate_sg_batch([test_q])
raw, clean = test_out[0]

print(f"Question:\n{test_q}")
print(f"\nRaw output:\n{raw}")
print(f"\nCleaned SG:\n{clean}")

steps = extract_steps(clean)
valid, reason = is_valid_sg(clean)
print(f"\nResult → valid={valid} | reason={reason} | steps={len(steps)}")
for s in steps:
    words = len(step_body(s).split())
    flag  = "✅" if words <= CONFIG["max_words_per_step"] else "❌ TOO LONG"
    print(f"  [{words:2d}w] {flag}  {s}")

print("\n" + "=" * 55)
print("If steps look clean and valid=True → run Cell 10")
print("=" * 55)

In [ ]:
# ── CELL 10: Main Loop ────────────────────────────────────────
# Only change from GSM8K version: use asdiv_question() / asdiv_answer()
# instead of b["question"] / b["answer"] (which was a GSM8K #### string).

print(f"Generating SG for {len(sampled_data)} ASDiv questions...")
print("-" * 50)

results   = []
start_idx = 0

# Resume from checkpoint if session died
if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["raw_output_file"]):
        with open(CONFIG["raw_output_file"]) as f:
            results = [json.loads(l) for l in f if l.strip()]
    print(f"Resumed: idx={start_idx}, saved={len(results)}")
else:
    print("Starting fresh...")

for batch_start in tqdm(
    range(start_idx, len(sampled_data), CONFIG["batch_size"]),
    desc="Generating"
):
    batch_end = min(batch_start + CONFIG["batch_size"], len(sampled_data))
    batch     = sampled_data[batch_start:batch_end]

    # ── ASDiv-specific: build question strings and extract clean answers ──
    questions = [asdiv_question(b) for b in batch]
    answers   = [asdiv_answer(b)   for b in batch]

    try:
        batch_out = generate_sg_batch(questions)
    except RuntimeError as e:
        print(f"OOM at batch {batch_start} — try reducing batch_size in Config")
        print(f"Error: {e}")
        break

    for i, (question, answer, (sg_raw, sg_clean)) in enumerate(
        zip(questions, answers, batch_out)
    ):
        valid, reason = is_valid_sg(sg_clean)
        steps         = extract_steps(sg_clean)

        results.append({
            "id"        : batch_start + i,
            "question"  : question,
            "sg_raw"    : sg_raw,
            "sg_clean"  : sg_clean,
            "sg_steps"  : steps,
            "gt_answer" : answer,   # already a clean numeric string
            "valid"     : valid,
            "reason"    : reason,
        })

    # Checkpoint
    if len(results) % CONFIG["save_every"] < CONFIG["batch_size"]:
        with open(CONFIG["raw_output_file"], "w") as f:
            for item in results: f.write(json.dumps(item) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": batch_end}, f)

# Final save
with open(CONFIG["raw_output_file"], "w") as f:
    for item in results: f.write(json.dumps(item) + "\n")

valid_count = sum(1 for r in results if r["valid"])
print(f"\nTotal     : {len(results)}")
print(f"Valid     : {valid_count}")
print(f"Invalid   : {len(results) - valid_count}")
print(f"Valid rate: {valid_count/max(1,len(results))*100:.1f}%")

In [ ]:
# ── CELL 11: Save + Show Samples ─────────────────────────────

valid_results = [r for r in results if r["valid"]]

with open(CONFIG["valid_output_file"], "w") as f:
    for item in valid_results:
        f.write(json.dumps(item) + "\n")

print(f"Saved {len(valid_results)} valid SG → {CONFIG['valid_output_file']}")
print("\n" + "=" * 60)
print("SAMPLE VALID SG OUTPUTS")
print("=" * 60)

for i, item in enumerate(valid_results[:5]):
    print(f"\n--- Example {i+1} ---")
    print(f"Q : {item['question'][:100]}")
    print("SG:")
    for s in item["sg_steps"]:
        words = len(step_body(s).split())
        print(f"  [{words:2d}w] {s}")
    print(f"GT: {item['gt_answer']}")

In [ ]:
# ── CELL 12: Report ───────────────────────────────────────────

print("=" * 60)
print("FINAL REPORT")
print("=" * 60)

valid_results = [r for r in results if r["valid"]]
failed        = [r for r in results if not r["valid"]]

step_counts = {}
for item in valid_results:
    n = len(item["sg_steps"])
    step_counts[n] = step_counts.get(n, 0) + 1

print(f"\nTotal generated : {len(results)}")
print(f"Valid           : {len(valid_results)} ({len(valid_results)/max(1,len(results))*100:.1f}%)")
print(f"Invalid         : {len(failed)}")

print("\nStep count distribution:")
for n, c in sorted(step_counts.items()):
    bar = "█" * max(1, c * 20 // max(1, len(valid_results)))
    print(f"  {n} steps: {c:3d} ({c/max(1,len(valid_results))*100:.0f}%) {bar}")

all_words = [
    len(step_body(s).split())
    for item in valid_results
    for s in item["sg_steps"]
]
if all_words:
    print(f"\nStep word count:")
    print(f"  Average : {sum(all_words)/len(all_words):.1f} words (target <{CONFIG['max_words_per_step']})")
    print(f"  Min/Max : {min(all_words)} / {max(all_words)} words")

reasons = {}
for r in failed:
    reasons[r["reason"]] = reasons.get(r["reason"], 0) + 1
if reasons:
    print(f"\nFailure breakdown:")
    for reason, count in sorted(reasons.items(), key=lambda x: -x[1]):
        print(f"  {reason:25s}: {count}")

report = {
    "model"          : CONFIG["model_name"],
    "dataset"        : "ASDiv",
    "total"          : len(results),
    "valid"          : len(valid_results),
    "valid_rate"     : round(len(valid_results)/max(1,len(results)), 4),
    "step_dist"      : step_counts,
    "avg_step_words" : round(sum(all_words)/max(1,len(all_words)), 1) if all_words else 0,
    "failure_reasons": reasons,
}
with open(CONFIG["report_file"], "w") as f:
    json.dump(report, f, indent=2)

print(f"\nReport → {CONFIG['report_file']}")
print("\n" + "=" * 60)
print("NEXT STEPS:")
print("=" * 60)
vr = len(valid_results)/max(1,len(results))*100
if vr >= 70:
    print(f"✅ Valid rate {vr:.0f}% — GOOD!")
    print("   1. Visually check 5 sample SGs above")
    print("   2. Add this notebook's output to Notebook 2 as input data")
    print("   3. Notebook 2 will find sg_valid.jsonl automatically")
elif vr >= 40:
    print(f"⚠️  Valid rate {vr:.0f}% — acceptable but not ideal")
else:
    print(f"❌ Valid rate {vr:.0f}% — check Cell 9 single test output")
print("=" * 60)